# ⚡ 따라쓰기 실습. 에너지 공공데이터 분석

**AI로 분석하는 신재생에너지 · Day 2 · 울산대학교**

한국에너지공단의 실제 공공데이터(전국 3,654행)로 **울산의 신재생에너지 현황**을 파악합니다.

### 📌 이 파일 사용법

1. 각 단계의 **회색 상자 안 코드**를 아래 빈 셀에 **직접 타이핑**합니다
2. `Shift + Enter` 로 실행합니다
3. **오늘은 함정이 3개 나옵니다 — 전부 일부러입니다.** 실제 공공데이터가 원래 이렇습니다

> 벅스 차트에서 쓴 순서 그대로: `read_csv` → `head` → `info` → 필터링. 데이터만 바뀝니다.

---
## 1단계. 파일 업로드

```python
from google.colab import files              # 코랩의 파일 업로드 도구

uploaded = files.upload()                   # 실행하면 [파일 선택] 버튼 등장

파일명 = list(uploaded.keys())[0]           # 방금 올린 파일의 이름을 자동으로 받아옴

print(파일명)                               # 어떤 이름으로 저장됐는지 눈으로 확인
```

💬 `울산대실습Day2_신재생에너지_보급_현황.csv` 를 선택하세요.

### ⚠️ 왜 파일명을 직접 안 치고 변수로 받나요?

한글 파일명은 **화면에 똑같이 보여도 컴퓨터 안에서는 다른 글자**일 수 있습니다.

(특히 맥에서 만든 파일은 한글 자모가 분리 저장됩니다 — 타이핑한 '현황'과 파일의 '현황'이 다른 바이트!)

그래서 눈으로 보고 똑같이 쳐도 `FileNotFoundError` 가 날 수 있습니다.

→ **이름을 변수에 받아 쓰면** 이 문제가 원천적으로 사라집니다. 실무에서도 이렇게 합니다.

> ⚠️ 코랩은 세션이 끊기면 파일이 사라집니다. `FileNotFoundError` 가 나오면 이 셀부터 다시!

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 2단계. ⚡ 함정 ① — 인코딩

```python
import pandas as pd                         # 표 데이터를 다루는 도구, 별명은 pd

df = pd.read_csv(파일명)                    # 벅스 때처럼 그냥 읽어보기
```

**빨간 에러가 나면 정상입니다.** 마지막 줄을 읽어보세요 → `UnicodeDecodeError`

(만약 `FileNotFoundError` 가 나오면 → 1단계 업로드부터 다시. 그건 함정이 아니라 파일이 없는 것!)

💬 한국 공공기관 파일은 옛날 한글 인코딩(**cp949**)을 씁니다. 공공데이터를 쓰는 한 평생 만나는 에러입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요 (에러가 나야 정상!)


### 해결 — 인코딩 지정

```python
df = pd.read_csv(파일명, encoding='cp949')  # 한글 인코딩 지정
```

💬 벅스 차트를 **저장**할 때 `utf-8-sig` 를 썼죠? 이번엔 **읽을 때** 인코딩입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 3단계. 건강검진 — 벅스와 같은 순서

```python
df.head()    # 위 5줄 미리보기 — 어떤 컬럼이 있나
```

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


```python
df.info()    # 행 수·컬럼·빈 값 확인
```

**✅ 확인할 것**
- `3654 entries` — 표가 이만하면 눈으로는 못 봅니다. 그래서 코드로 봅니다
- 빈 값(Null)은 거의 없음 — 정부 통계라서 깨끗한 편입니다

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 4단계. ⚡ 함정 ② — 숨은 공백

```python
df['발전량(MWh)']                            # 발전량 컬럼만 꺼내보기
```

**또 에러!** 마지막 줄 → `KeyError`. 분명히 head()에서 봤던 이름인데 왜 없다고 할까요?

### 범인 찾기

```python
df.columns                                  # 컬럼 이름을 '정확하게' 확인
```

출력을 자세히 보면 → `' 발전량(MWh) '` — **이름 앞뒤에 공백**이 숨어 있습니다!

💬 사람 눈에는 안 보이지만 컴퓨터에게 `'발전량'` 과 `' 발전량 '` 은 완전히 다른 이름입니다.

In [ ]:
# ✏️ 위 코드 두 줄을 차례로 실행해 보세요 (첫 줄은 에러가 나야 정상!)


### 해결 — 공백 청소

```python
df.columns = df.columns.str.strip()         # 모든 컬럼 이름의 앞뒤 공백 제거

df['발전량(MWh)']                            # 이제 정상 작동
```

💬 `strip` = 양끝 공백 벗기기.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 5단계. 지역별 합계 — 그런데 뭔가 이상하다?

벅스에서 최애를 찾던 필터링, 여기서는 **groupby(묶어서 계산)**로 확장합니다.

```python
df.groupby('광역')['발전량(MWh)'].sum().sort_values()   # 광역시도별 발전량 합계를 작은 순으로
```

### 🤔 잠깐 — 울산이 약 450만 MWh?

Day 1에서 울산은 **약 75만 MWh(750GWh)** 수준이라고 했는데, **6배**가 나왔습니다.

**컴퓨터는 계산을 틀리지 않습니다. 그럼 무엇이 틀렸을까요?**

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 6단계. 범인 찾기 — 데이터 구조 파악

```python
df['에너지원'].unique()                      # 에너지원 컬럼에 어떤 값들이 있나 (중복 제거)
```

**✅ 출력을 보면** — 태양광·풍력 같은 **세부 에너지원** 사이에
`'신·재생에너지'`, `'재생에너지'`, `'신에너지'` 라는 **합계 분류**가 섞여 있습니다!
(Day 1의 그 법적 분류가 데이터에 그대로 들어 있네요 — 재생 + 신 = 신재생)

```python
df[df['광역'] == df['기초']].head()          # 광역과 기초가 같은 행 = 시도 전체 합계 행
```

**✅ 그리고** — '서울/서울', '부산/부산' 같은 **시도 합계 행**도 구·군 세부 행과 섞여 있습니다.

💬 합계 행 + 세부 행을 전부 더했으니 **같은 발전량이 여러 번 계산**된 것.

> 📌 오늘의 교훈: **데이터 구조를 모르고 집계하면, 틀린 답이 그럴듯하게 나온다.**

In [ ]:
# ✏️ 위 코드 두 개를 차례로 실행하며 구조를 확인하세요


---
## 7단계. 제대로 계산하기 — 필터 후 groupby

```python
시도별 = df[df['광역'] == df['기초']]                                  # ① 시도 합계 행만 남기기

세부 = 시도별[~시도별['에너지원'].isin(['신·재생에너지', '재생에너지', '신에너지'])]   # ② 합계 분류 제외 (~는 '반대')

지역별 = 세부.groupby('광역')['발전량(MWh)'].sum().sort_values()        # ③ 이제 제대로 집계

지역별                                                                # 결과 확인
```

**✅ 확인** — 울산이 약 `750,479 MWh` (= 750 GWh) 근처면 일단 성공!

### 🤔 그런데... 시도가 몇 개 보입니까?

```python
len(지역별)                                 # 시도 개수 세기
```

**16개?!** 대한민국 광역시도는 **17개**입니다. 누가 사라졌을까요?

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## 8단계. ⚡ 함정 ③ — 제주 실종 사건

```python
df[df['광역'] == '제주']['기초'].unique()    # 제주의 '기초' 값들을 확인해 보자
```

**✅ 출력** → `['제주도', '제주시', '서귀포시', '기타']`

범인 발견! 제주의 합계 행은 기초가 `'제주'` 가 아니라 **`'제주도'`** 로 적혀 있습니다.

그래서 `광역 == 기초` 조건(제주 == 제주도?)에 안 걸려서 통째로 빠진 겁니다.

### 해결 — 조건에 '또는'을 추가

```python
시도별 = df[(df['광역'] == df['기초']) | (df['기초'] == '제주도')]      # | 는 '또는' — 제주도 합계 행도 포함

세부 = 시도별[~시도별['에너지원'].isin(['신·재생에너지', '재생에너지', '신에너지'])]   # 합계 분류 제외 (아까와 동일)

지역별 = 세부.groupby('광역')['발전량(MWh)'].sum().sort_values()        # 다시 집계

print(len(지역별))                           # 이제 17개!

지역별                                       # 최종 순위 확인
```

> 📌 함정 3개의 공통 교훈 — 인코딩도, 공백도, 이름도: **데이터는 눈으로 확인하기 전까지 믿지 않는다.**

### 🤔 최종 결과를 읽어 보기

1. 울산은 작은 쪽에서 **몇 번째**입니까?
2. 1위 지역과 울산의 차이는 **몇 배**쯤 됩니까?
3. **제주**는 울산보다 큽니까 작습니까? — 내일 제주 이야기가 나오는 이유입니다

In [ ]:
# ✏️ 위 코드 두 개를 차례로 실행하세요


---
## 9단계. 🎯 미션 — 울산은 무엇으로 전기를 만드나

벅스에서 `df[조건]` 으로 최애를 찾았죠. 이번엔 **울산**을 찾습니다.

```python
울산 = 세부[세부['광역'] == '울산']                                    # 울산 행만 골라내기

울산.sort_values('발전량(MWh)', ascending=False)[['에너지원', '발전량(MWh)']]   # 발전량 큰 순으로 정렬해서 보기
```

### 🤔 결과를 읽어 보기

1. 울산의 **1위 에너지원**은 무엇입니까? 예상과 같았나요?
2. 값이 **0인 에너지원**들도 눈여겨 보세요 — 없다는 것도 정보입니다
3. 이 구성을 보면 울산은 어떤 도시라고 말할 수 있을까요?

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


### 미니 검산 — 정부 통계도 검산할 수 있다

```python
울산['발전량(MWh)'].sum()                    # 세부 에너지원을 직접 전부 더하기
```

**✅ 확인** — 8단계 순위표의 울산 값과 같은 숫자(750,479)가 나오면, 우리의 계산 경로가 옳았다는 뜻입니다.

In [ ]:
# ✏️ 위 코드를 여기에 따라 입력하세요


---
## ✅ 오늘 쓴 함수 정리

| 함수 | 하는 일 |
|---|---|
| `pd.read_csv(파일, encoding='cp949')` | 한글 공공데이터 읽기 |
| `df.columns.str.strip()` | 컬럼 이름 공백 청소 |
| `df['컬럼'].unique()` | 어떤 값들이 있나 (중복 제거) |
| `df[조건]` / `~` / `\|` / `.isin([...])` | 필터링 — 남기기 / 반대 / 또는 / 목록 중 하나 |
| `df.groupby('기준')['값'].sum()` | 기준별로 묶어서 합계 |
| `sort_values()` / `len()` | 정렬 / 개수 세기 |

### ⚡ 오늘 만난 함정 3개 — 이제 안 무섭죠?

| 함정 | 정체 | 해결 |
|---|---|---|
| ① `UnicodeDecodeError` | 한글 인코딩 | `encoding='cp949'` |
| ② `KeyError` | 컬럼명 숨은 공백 | `df.columns.str.strip()` |
| ③ 제주 실종 | 합계 행 이름 불일치('제주도') | 조건에 `\|` (또는) 추가 |

### ⏭️ 다음 시간 예고 — 시각화

오늘 만든 `지역별` 과 `울산` 변수를 **그대로 그래프로** 그립니다. (변수 지우지 마세요!)

숫자 750,479보다 막대 하나가 더 많은 것을 말해 줍니다.